In [4]:
!pip install tf-levenberg-marquardt 

model_12_6_0.xlsx', model_13_9_6.xlsx', model_20_5_0.xlsx', model_22_8_7.xlsx', model_24_6_2.xlsx', model_2_8_0.xlsx', model_30_7_8.xlsx', model_9_9_10.xlsx', model_9_9_2.xlsx', model_9_9_5.xlsx'

In [5]:
import pandas as pd
import numpy as np
from sklearn.metrics import r2_score

def redes(file_names):
    df_list = [pd.read_excel(arquivo) for arquivo in file_names]
    z_preds = [df['Z_pred'].values.reshape(-1, 1) for df in df_list]
    Z_pred_total = np.hstack(z_preds)
    Z_pred_sum = np.sum(Z_pred_total, axis=1).reshape(-1, 1)
    df = pd.read_excel("1000_model/1000_model_12_6_0.xlsx")
    Z = df['Z'].values.reshape(-1, 1)

    mse_sup = np.mean((Z - Z_pred_sum/10) ** 2)
    r2_sup = r2_score(Z, Z_pred_sum)

    print(f"mse: {mse_sup}")
    return mse_sup, r2_sup

file_names= ["1000_model/1000_model_12_6_0.xlsx", "1000_model/1000_model_13_9_6.xlsx",
              "1000_model/1000_model_20_5_0.xlsx", "1000_model/1000_model_22_8_7.xlsx", 
              "1000_model/1000_model_24_6_2.xlsx", "1000_model/1000_model_2_8_0.xlsx", 
              "1000_model/1000_model_30_7_8.xlsx","1000_model/1000_model_9_9_0.xlsx",
              "1000_model/1000_model_9_9_2.xlsx", "1000_model/1000_model_9_9_5.xlsx"]
result= redes(file_names)




mse: 0.6795612726992786


In [22]:
import pandas as pd
import numpy as np
from sklearn.metrics import r2_score
from sklearn.preprocessing import StandardScaler
import matplotlib.pyplot as plt


lambda_reg = 5e-6        
lr = 3e-5                 
n_epocas = 5000000


def redes(file_names):
    df_list = [pd.read_excel(arquivo) for arquivo in file_names]
    z_preds = np.hstack([df['Z_pred'].values.reshape(-1,1) for df in df_list])
    dfZ = pd.read_excel("25_model/25_model_12_6_0.xlsx")
    Z = dfZ['Z'].values.reshape(-1, 1)
    N = Z.shape[0]
    return Z, z_preds, N

def pesos(Z, z_preds, N, patience=500, min_delta=10e-3):  
    
    best_erro = np.inf
    best_w = None
    patience_counter = 0
    a = np.random.uniform(0, 1, z_preds.shape[1])

    for epoch in range(1, n_epocas+1):
        w = np.exp(a) / np.sum(np.exp(a)) # softmax
        yhat = np.dot(z_preds, w)
        residuo = Z.flatten() - yhat
        mse = np.mean(residuo**2)
        mse_pond = mse + lambda_reg * np.sum(a**2)
        # gradiente dmse/dw
        gmse = (-2.0 / N) * z_preds.T.dot(residuo)
        # gradiente completo
        s = np.dot(gmse, w)
        grad_a = w * (gmse - s) + 2.0 * lambda_reg * a
        # atualização
        a = a - lr * grad_a

        # ---- EARLY STOPPING ----
        if mse_pond < best_erro - min_delta:
            best_erro = mse_pond
            best_w = w.copy()
            patience_counter = 0  # reset
        else:
            patience_counter += 1

        if patience_counter >= patience:
            print(f"\nEarly stopping ativado na época {epoch} — perda não melhorou por {patience} épocas.")
            break
        # -------------------------

        if epoch % 100 == 0 or epoch == 1:
            r2 = 1 - np.sum((Z - yhat.reshape(-1,1))**2) / np.sum((Z - Z.mean())**2)
            print(f"epoch {epoch:4d} loss_25={mse_pond:.6f} mse_25={mse:.6f} R2_25={r2:.4f} weights={w}")

    return best_erro, best_w

def mse(file_names_1000, best_w):
    df_list = [pd.read_excel(arquivo) for arquivo in file_names_1000]
    z_preds = [df['Z_pred'].values.reshape(-1, 1) for df in df_list]
    Z_pred_total = np.hstack(z_preds)
    yhat_final = Z_pred_total * best_w      
    Z_pred_sum = np.sum(yhat_final, axis=1).reshape(-1, 1)
    df = pd.read_excel("1000_model/1000_model_12_6_0.xlsx")
    Z = df['Z'].values.reshape(-1, 1)

    mse_sup = np.mean((Z - Z_pred_sum) ** 2)
    r2_sup = r2_score(Z, Z_pred_sum)

    print(f"MSE conjunto = {mse_sup}")
    print(f"R² conjunto = {r2_sup}")

    return mse_sup, r2_sup

file_names_1000 =  ["1000_model/1000_model_12_6_0.xlsx", "1000_model/1000_model_13_9_6.xlsx",
              "1000_model/1000_model_20_5_0.xlsx", "1000_model/1000_model_22_8_7.xlsx", 
              "1000_model/1000_model_24_6_2.xlsx", "1000_model/1000_model_2_8_0.xlsx", 
              "1000_model/1000_model_30_7_8.xlsx","1000_model/1000_model_9_9_0.xlsx",
              "1000_model/1000_model_9_9_2.xlsx", "1000_model/1000_model_9_9_5.xlsx"]

file_names = [
    "25_model/25_model_12_6_0.xlsx",
    "25_model/25_model_13_9_6.xlsx",
    "25_model/25_model_20_5_0.xlsx",
    "25_model/25_model_22_8_7.xlsx",
    "25_model/25_model_24_6_2.xlsx",
    "25_model/25_model_2_8_0.xlsx",
    "25_model/25_model_30_7_8.xlsx",
    "25_model/25_model_9_9_0.xlsx",
    "25_model/25_model_9_9_2.xlsx",
    "25_model/25_model_9_9_5.xlsx"
]

Z, z_preds, N= redes(file_names)
best_erro, best_w = pesos(Z, z_preds, N)

print("best erro =", best_erro)
print("best pesos =", best_w)
mse_1000 =mse(file_names_1000, best_w)



epoch    1 loss_25=0.025876 mse_25=0.025863 R2_25=0.9938 weights=[0.08861121 0.07296197 0.10301226 0.07876094 0.07198065 0.15987686
 0.15493181 0.07062303 0.12816965 0.07107161]
epoch  100 loss_25=0.025875 mse_25=0.025862 R2_25=0.9938 weights=[0.0886118  0.07296266 0.10301214 0.07876184 0.07198134 0.15987695
 0.15493266 0.07062232 0.12816735 0.07107095]
epoch  200 loss_25=0.025875 mse_25=0.025862 R2_25=0.9938 weights=[0.08861238 0.07296335 0.10301203 0.07876274 0.07198202 0.15987704
 0.15493352 0.07062161 0.12816502 0.07107028]
epoch  300 loss_25=0.025875 mse_25=0.025862 R2_25=0.9938 weights=[0.08861297 0.07296405 0.10301191 0.07876364 0.07198271 0.15987713
 0.15493438 0.0706209  0.12816269 0.07106961]
epoch  400 loss_25=0.025875 mse_25=0.025861 R2_25=0.9938 weights=[0.08861356 0.07296474 0.10301179 0.07876455 0.0719834  0.15987723
 0.15493524 0.07062019 0.12816037 0.07106894]
epoch  500 loss_25=0.025874 mse_25=0.025861 R2_25=0.9938 weights=[0.08861415 0.07296544 0.10301167 0.07876545 

In [31]:
lambda_reg = 5e-6        
lr = 1e-2                 
n_epocas = 5000000

best_erro1, best_w_1 = pesos(Z, z_preds, N)

print("best erro =", best_erro1)
print("best pesos =", best_w_1)
mse_1000 =mse(file_names_1000, best_w_1)

epoch    1 loss_25=0.031434 mse_25=0.031421 R2_25=0.9925 weights=[0.08528035 0.08507849 0.07097356 0.08018393 0.09355268 0.13949888
 0.06692309 0.13358741 0.10029813 0.14462349]
epoch  100 loss_25=0.031221 mse_25=0.031208 R2_25=0.9926 weights=[0.0855697  0.08552793 0.0710519  0.08060996 0.09407292 0.13984756
 0.06707987 0.13271657 0.09984292 0.14368068]
epoch  200 loss_25=0.031009 mse_25=0.030996 R2_25=0.9926 weights=[0.08585794 0.0859807  0.07112762 0.08103959 0.09459707 0.14019036
 0.0672356  0.13184679 0.09938558 0.14273876]
epoch  300 loss_25=0.030799 mse_25=0.030786 R2_25=0.9927 weights=[0.08614217 0.08643225 0.07119995 0.08146852 0.09511987 0.1405238
 0.06738872 0.1309868  0.09893074 0.1418072 ]
epoch  400 loss_25=0.030592 mse_25=0.030579 R2_25=0.9927 weights=[0.08642238 0.08688258 0.07126894 0.08189678 0.09564131 0.14084796
 0.06753925 0.13013647 0.09847845 0.14088588]
epoch  500 loss_25=0.030387 mse_25=0.030374 R2_25=0.9928 weights=[0.08669859 0.0873317  0.07133465 0.08232436 0

In [109]:
lambda_reg = 5e-6        
lr = 1e-2                 
n_epocas = 5000000

best_erro1, best_w_1 = pesos(Z, z_preds, N)

print("best erro =", best_erro1)
print("best pesos =", best_w_1)
mse_1000 =mse(file_names_1000, best_w_1)

epoch    1 loss_25=0.032543 mse_25=0.032531 R2_25=0.9922 weights=[0.06881172 0.07481899 0.10033723 0.11122965 0.08767741 0.0863295
 0.07666262 0.07991302 0.15898433 0.15523553]
epoch  100 loss_25=0.032290 mse_25=0.032278 R2_25=0.9923 weights=[0.06905479 0.07520124 0.10043401 0.11204755 0.08817791 0.08661219
 0.07688896 0.07964212 0.15778663 0.1541546 ]
epoch  200 loss_25=0.032037 mse_25=0.032026 R2_25=0.9924 weights=[0.06929655 0.07558486 0.10052417 0.11287417 0.08868091 0.08689275
 0.0771134  0.07936783 0.15659138 0.15307397]
epoch  300 loss_25=0.031788 mse_25=0.031777 R2_25=0.9924 weights=[0.06953453 0.07596598 0.10060683 0.11370124 0.08918132 0.08716833
 0.07733367 0.07909298 0.1554106  0.15200453]
epoch  400 loss_25=0.031543 mse_25=0.031532 R2_25=0.9925 weights=[0.06976874 0.07634461 0.10068209 0.11452876 0.08967913 0.08743897
 0.07754978 0.07881764 0.15424408 0.15094621]
epoch  500 loss_25=0.031300 mse_25=0.031289 R2_25=0.9925 weights=[0.0699992  0.07672073 0.10075008 0.11535673 0